In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from scipy.sparse import hstack, csr_matrix

nltk.download("stopwords")

# 2. Load data
df = pd.read_csv("online_shoppers_intention2.csv")
print(df.shape)
display(df.head())

# 3. Find text column
text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Text columns:", text_cols)

# Select first text column
text_col = text_cols[0] if text_cols else None
print("Using:", text_col)

# 4. NLP cleaning
stop_words = set(stopwords.words("english"))

def clean_text(x):
    x = str(x).lower()
    x = re.sub(r"[^a-zA-Z\s]", " ", x)
    return " ".join(w for w in x.split() if w not in stop_words)

if text_col:
    df["clean_text"] = df[text_col].fillna("").apply(clean_text)

    tfidf = TfidfVectorizer(max_features=300)
    X_text = tfidf.fit_transform(df["clean_text"])
else:
    X_text = None

# 5. Numerical features
num_cols = df.select_dtypes(include=np.number).columns.tolist()

# Don't use Revenue as a clustering feature
if "Revenue" in num_cols:
    num_cols.remove("Revenue")

scaler = StandardScaler()
X_num = scaler.fit_transform(df[num_cols])

# 6. Combine numerical + NLP features
X = csr_matrix(X_num)

if X_text is not None:
    X = hstack([X, X_text])

print("Final features:", X.shape)

# 7. Find best K
scores = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    scores.append(silhouette_score(X, labels))

best_k = range(2, 7)[np.argmax(scores)]
print("Best K:", best_k)
print("Silhouette Score:", max(scores))

# 8. K-Means
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X)

# 9. Cluster results
print(df["Cluster"].value_counts().sort_index())
display(df.head())

# 10. PCA visualization
X_pca = PCA(n_components=2).fit_transform(X.toarray())

plt.figure(figsize=(8,5))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df["Cluster"], cmap="viridis")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("K-Means Clusters")
plt.colorbar(label="Cluster")
plt.show()

# 11. Cluster summary
display(df.groupby("Cluster")[num_cols].mean())

# 12. Save results
df.to_csv("online_shoppers_intention2_clustered.csv", index=False)
print("Saved successfully!")

(12330, 18)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adnanaltimeemy/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


Text columns: ['Month', 'VisitorType']
Using: Month
Final features: (12330, 24)
